# BioFuse Tutorial 2: Adding a New Dataset

This tutorial shows you how to integrate your own custom dataset into BioFuse.

We'll cover three approaches:
1. **Quick approach**: Using existing loaders for directory-based datasets
2. **Custom PyTorch Dataset**: Creating a dataset class
3. **Full integration**: Adding a new loader to BioFuse

## Example Dataset

We'll use a hypothetical skin lesion dataset as an example.

## Approach 1: Quick Start with Directory-Based Datasets

If your dataset is organized as:
```
dataset/
├── train/
│   ├── class1/
│   │   ├── img1.jpg
│   │   └── img2.jpg
│   └── class2/
│       ├── img3.jpg
│       └── img4.jpg
└── test/
    └── ...
```

You can use BioFuse's built-in `load_custom_directory` function!

In [ ]:
from biofuse.data import load_custom_directory
from biofuse import BioFuse, get_classifier, compute_metrics
from pathlib import Path

# Path to your dataset
dataset_root = Path('/path/to/your/dataset')

# Load train and test sets
train_data, num_classes = load_custom_directory(
    root=dataset_root / 'train',
    img_size=224,
    extension='.jpg'  # or '.png', '.jpeg', etc.
)

test_data, _ = load_custom_directory(
    root=dataset_root / 'test',
    img_size=224,
    extension='.jpg'
)

print(f"Number of classes: {num_classes}")
print(f"Train samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")

In [ ]:
# Now use BioFuse as normal
biofuse = BioFuse(models=['BioMedCLIP'], fusion_method='concat')

# Extract embeddings
train_emb, train_labels = biofuse.extract_embeddings_from_loader(train_data)
test_emb, test_labels = biofuse.extract_embeddings_from_loader(test_data)

# Train classifier
clf = get_classifier('logistic', num_classes=num_classes)
clf.fit(train_emb, train_labels)

# Evaluate
pred = clf.predict(test_emb)
proba = clf.predict_proba(test_emb)
metrics = compute_metrics(test_labels, pred, proba, num_classes, task='multi-class')

print(f"Accuracy: {metrics['accuracy']:.4f}")

## Approach 2: Custom PyTorch Dataset

For more control, create a custom PyTorch Dataset class.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms
import pandas as pd
from pathlib import Path

class SkinLesionDataset(Dataset):
    """
    Custom dataset for skin lesion classification.
    
    Expected structure:
    - CSV file with columns: 'image_path', 'label'
    - Images in a directory
    """
    
    def __init__(self, csv_file, img_dir, img_size=224, transform=None):
        """
        Args:
            csv_file (str): Path to CSV with image paths and labels
            img_dir (str): Directory with all images
            img_size (int): Target image size
            transform (callable, optional): Optional transform
        """
        self.df = pd.read_csv(csv_file)
        self.img_dir = Path(img_dir)
        self.img_size = img_size
        
        # Default transform if none provided
        if transform is None:
            self.transform = transforms.Compose([
                transforms.Resize((img_size, img_size)),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]
                )
            ])
        else:
            self.transform = transform
        
        # Create label mapping
        self.label_names = sorted(self.df['label'].unique())
        self.label_to_idx = {label: idx for idx, label in enumerate(self.label_names)}
        self.num_classes = len(self.label_names)
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        # Get image path and label
        img_name = self.df.iloc[idx]['image_path']
        label_name = self.df.iloc[idx]['label']
        
        # Load image
        img_path = self.img_dir / img_name
        image = Image.open(img_path).convert('RGB')
        
        # Apply transform
        if self.transform:
            image = self.transform(image)
        
        # Convert label to index
        label = self.label_to_idx[label_name]
        
        return image, label


# Example usage
train_dataset = SkinLesionDataset(
    csv_file='/path/to/train.csv',
    img_dir='/path/to/images',
    img_size=224
)

test_dataset = SkinLesionDataset(
    csv_file='/path/to/test.csv',
    img_dir='/path/to/images',
    img_size=224
)

print(f"Number of classes: {train_dataset.num_classes}")
print(f"Class names: {train_dataset.label_names}")
print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

In [ ]:
# Use with BioFuse
from biofuse import BioFuse

biofuse = BioFuse(models=['BioMedCLIP', 'CONCH'], fusion_method='concat')

# Extract embeddings
train_emb, train_labels = biofuse.extract_embeddings_from_loader(
    DataLoader(train_dataset, batch_size=32, shuffle=False)
)

test_emb, test_labels = biofuse.extract_embeddings_from_loader(
    DataLoader(test_dataset, batch_size=32, shuffle=False)
)

print(f"Train embeddings: {train_emb.shape}")
print(f"Test embeddings: {test_emb.shape}")

## Approach 3: Full Integration into BioFuse

For datasets you'll use frequently, add them to BioFuse's data module.

### Step 1: Create a loader function in `biofuse/data/loaders.py`

In [ ]:
# Add this to biofuse/data/loaders.py

"""
def load_skin_lesion(
    split: str = 'train',
    img_size: int = 224,
    data_root: Optional[Union[str, Path]] = None,
    download: bool = False
) -> Tuple[Dataset, int]:
    \"\"\"Load skin lesion dataset.
    
    Args:
        split: One of 'train', 'val', 'test'
        img_size: Target image size
        data_root: Root directory for dataset
        download: Whether to download dataset if not present
    
    Returns:
        dataset: PyTorch Dataset
        num_classes: Number of classes
    \"\"\"    
    if data_root is None:
        data_root = Path.home() / 'data' / 'skin_lesion'
    else:
        data_root = Path(data_root)
    
    # Download logic (if applicable)
    if download and not data_root.exists():
        _download_skin_lesion(data_root)
    
    # CSV file paths
    csv_file = data_root / f'{split}.csv'
    img_dir = data_root / 'images'
    
    # Create dataset
    dataset = SkinLesionDataset(
        csv_file=csv_file,
        img_dir=img_dir,
        img_size=img_size
    )
    
    return dataset, dataset.num_classes


def _download_skin_lesion(data_root: Path):
    \"\"\"Download skin lesion dataset.\"\"\"    
    # Implement download logic here
    # Example: wget, requests, kaggle API, etc.
    pass
"""

print("Code to add to biofuse/data/loaders.py shown above")

### Step 2: Export in `biofuse/data/__init__.py`

In [ ]:
# Add to biofuse/data/__init__.py

"""
from .loaders import (
    load_medmnist,
    load_imagenet,
    load_busi,
    load_skin_lesion,  # <-- Add this
    load_custom_directory,
    create_custom_dataset,
)

__all__ = [
    'load_medmnist',
    'load_imagenet',
    'load_busi',
    'load_skin_lesion',  # <-- Add this
    'load_custom_directory',
    'create_custom_dataset',
    'BioFuseImageDataset',
]
"""

print("Code to add to biofuse/data/__init__.py shown above")

### Step 3: Update main `biofuse/__init__.py` for lazy loading

In [ ]:
# Add to biofuse/__init__.py in the __getattr__ function

"""
def __getattr__(name):
    # ... existing code ...
    
    # Data
    if name == 'load_skin_lesion':  # <-- Add this
        from .data import load_skin_lesion
        return load_skin_lesion
    
    # ... rest of code ...

__all__ = [
    # ... existing exports ...
    'load_skin_lesion',  # <-- Add this
]
"""

print("Code to add to biofuse/__init__.py shown above")

### Step 4: Update CLI to support new dataset

In [ ]:
# In biofuse/cli/train.py, update the dataset loading logic

"""
# Around line 150-200, add to the dataset loading section:

elif dataset_name == 'skin_lesion':
    from biofuse.data import load_skin_lesion
    train_data, num_classes = load_skin_lesion(
        split='train',
        img_size=img_size,
        download=True
    )
    val_data, _ = load_skin_lesion(
        split='val',
        img_size=img_size
    )
    test_data, _ = load_skin_lesion(
        split='test',
        img_size=img_size
    )
"""

print("Code to add to biofuse/cli/train.py shown above")

### Step 5: Use it!

Now you can use your dataset just like MedMNIST:

In [ ]:
# Python API
from biofuse import load_skin_lesion, BioFuse, get_classifier

train_data, num_classes = load_skin_lesion('train')
test_data, _ = load_skin_lesion('test')

biofuse = BioFuse(models=['BioMedCLIP'])
train_emb, train_labels, _, _, _ = biofuse.generate_embeddings(
    train_data=None,
    dataset_type='skin_lesion',
    dataset_name='skin_lesion'
)

# ... train and evaluate

In [ ]:
# CLI
!biofuse train --dataset skin_lesion --models BioMedCLIP,CONCH --classifier xgboost

## Real-World Example: Adding ISIC Dataset

Here's a complete example for the ISIC skin lesion dataset:

In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as transforms
import pandas as pd
from pathlib import Path
from typing import Tuple, Optional, Union

class ISICDataset(Dataset):
    """ISIC Skin Lesion Dataset."""
    
    CLASSES = ['melanoma', 'nevus', 'basal_cell_carcinoma', 
               'actinic_keratosis', 'benign_keratosis', 
               'dermatofibroma', 'vascular_lesion']
    
    def __init__(self, csv_file, img_dir, img_size=224):
        self.df = pd.read_csv(csv_file)
        self.img_dir = Path(img_dir)
        self.img_size = img_size
        self.num_classes = len(self.CLASSES)
        
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.img_dir / f"{row['image_name']}.jpg"
        image = Image.open(img_path).convert('RGB')
        image = self.transform(image)
        label = self.CLASSES.index(row['diagnosis'])
        return image, label


def load_isic(
    split: str = 'train',
    img_size: int = 224,
    data_root: Optional[Union[str, Path]] = None
) -> Tuple[Dataset, int]:
    """Load ISIC dataset."""
    if data_root is None:
        data_root = Path.home() / 'data' / 'isic'
    else:
        data_root = Path(data_root)
    
    csv_file = data_root / f'{split}.csv'
    img_dir = data_root / 'images'
    
    dataset = ISICDataset(csv_file, img_dir, img_size)
    return dataset, dataset.num_classes


# Test it
# train_data, num_classes = load_isic('train')
# print(f"ISIC dataset loaded: {len(train_data)} samples, {num_classes} classes")

## Summary

You now know three ways to add datasets to BioFuse:

1. **Quick Start**: Use `load_custom_directory()` for simple folder structures
2. **Custom Dataset**: Create a PyTorch Dataset class for more control
3. **Full Integration**: Add to BioFuse's data module for reusability

### Checklist for Adding a Dataset

- [ ] Create PyTorch Dataset class
- [ ] Implement `__len__` and `__getitem__`
- [ ] Add proper transforms (resize, normalize)
- [ ] Create loader function in `biofuse/data/loaders.py`
- [ ] Export in `biofuse/data/__init__.py`
- [ ] Add to main `biofuse/__init__.py` lazy loading
- [ ] Update CLI in `biofuse/cli/train.py`
- [ ] Test with Python API and CLI
- [ ] Add example config to `examples/configs/`

## Next Tutorial

**Tutorial 3**: Learn how to add new foundation models to BioFuse